In [50]:
%load_ext autoreload
%autoreload 2

import plotly.express as px
import torch
import torch.optim as optim
from models.custom_net_v2 import Net
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    MulticlassPrecision,
    MulticlassRecall,
)
from train import Trainer
from utils import get_datasets_and_loaders

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
classes = ("buildings", "forest", "glacier", "mountain", "sea", "street")

train_set, train_loader, valid_set, valid_loader = get_datasets_and_loaders()

In [52]:
image, label = next(iter(train_loader))
image = image[0:8]
image = image.permute(0, 2, 3, 1)
fig = px.imshow(image, facet_col=0, facet_col_spacing=0.01, height=300)
for i in range(image.shape[0]):
    fig.update_annotations(
        {"text": classes[label[i]]}, selector={"text": f"facet_col={i}"}
    )
    # fig.layout.annotations[i]["text"] = classes[label[i]]
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

In [57]:
num_epochs = 30

model = Net()

new_params = []
old_params = []
for name, param in model.named_parameters():
    if not any(sub in name for sub in ["final_linear", "conv1x1_2"]):
        old_params.append(param)
    else:
        new_params.append(param)

loaders = {"train_loader": train_loader, "valid_loader": valid_loader}

optimizer = optim.Adam(
    [{"params": old_params, "lr": 1e-6}, {"params": new_params, "lr": 1e-4}],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[1e-5, 1e-3], total_steps=len(train_loader) * num_epochs
)
optimization = {
    "criterion": nn.CrossEntropyLoss(),
    "optimizer": optimizer,
    "scheduler": scheduler,
}

metrics = MetricCollection(
    {
        "Accuracy": MulticlassAccuracy(num_classes=6, average="micro"),
        "Precision": MulticlassPrecision(num_classes=6, average="macro"),
        "Recall": MulticlassRecall(num_classes=6, average="macro"),
        "F1": MulticlassF1Score(num_classes=6, average="macro"),
    }
)
conf_matrix = MulticlassConfusionMatrix(num_classes=6)
log = {"metrics": metrics, "conf_matrix": conf_matrix, "writer": SummaryWriter()}

config = {
    "checkpoint_path": "checkpoints/",
    "classes": classes,
}

trainer = Trainer(model, loaders, optimization, log, config)
trainer.load_checkpoint(path="checkpoints/best_model_checkpoint.pth", verbose=False);

In [ ]:
trainer.fit(
    num_epochs=num_epochs,
    start_epoch=trainer.load_checkpoint(path="checkpoints/best_model_checkpoint.pth"),
)

## TODO

1. Оценить производительность модели
2. Импортировать ResNet50 и сделать fine-tuning